# UK Wind Forecast & Reliability Analysis

This notebook analyses:
1. **Forecast error characteristics** – mean, median, p99, error vs forecast horizon, error by time of day.
2. **Wind power reliability** – distribution of actual generation and a recommendation for how many MW can be reliably expected to meet demand.

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_style("whitegrid")
BASE = "https://data.elexon.co.uk/bmrs/api/v1"

## 1. Load data

Fetch actual wind generation (FUELHH, WIND) and wind forecasts (WINDFOR) for January 2024.

In [ ]:
# Actual generation
r_actual = requests.get(
    f"{BASE}/datasets/FUELHH/stream",
    params={"settlementDateFrom": "2024-01-01", "settlementDateTo": "2024-01-31", "fuelType": "WIND"},
    headers={"Accept": "application/json"},
)
actual_raw = r_actual.json() if r_actual.ok else []
# Normalize field names
actual_list = []
for row in actual_raw:
    st = row.get("startTime") or row.get("settlementStartTime", "")
    gen = row.get("generation") if "generation" in row else row.get("quantity", 0)
    actual_list.append({"startTime": st, "generation": float(gen) if gen is not None else np.nan})
df_actual = pd.DataFrame(actual_list)
df_actual["startTime"] = pd.to_datetime(df_actual["startTime"])
df_actual = df_actual.dropna(subset=["generation"]).sort_values("startTime")
print("Actual generation shape:", df_actual.shape)
df_actual.head()

In [ ]:
# Forecasts
r_forecast = requests.get(
    f"{BASE}/datasets/WINDFOR/stream",
    params={"publishDateTimeFrom": "2024-01-01T00:00:00Z", "publishDateTimeTo": "2024-01-31T23:59:59Z"},
    headers={"Accept": "application/json"},
)
forecast_raw = r_forecast.json() if r_forecast.ok else []
forecast_list = []
for row in forecast_raw:
    st = row.get("startTime") or row.get("settlementStartTime", "")
    pt = row.get("publishTime") or row.get("publishDateTime", "")
    gen = row.get("generation") if "generation" in row else row.get("quantity", 0)
    forecast_list.append({"startTime": st, "publishTime": pt, "generation": float(gen) if gen is not None else np.nan})
df_forecast = pd.DataFrame(forecast_list)
df_forecast["startTime"] = pd.to_datetime(df_forecast["startTime"])
df_forecast["publishTime"] = pd.to_datetime(df_forecast["publishTime"])
df_forecast = df_forecast.dropna(subset=["generation"])
# Horizon in hours (0–48)
df_forecast["horizon_hours"] = (df_forecast["startTime"] - df_forecast["publishTime"]).dt.total_seconds() / 3600
df_forecast = df_forecast[(df_forecast["horizon_hours"] >= 0) & (df_forecast["horizon_hours"] <= 48)]
print("Forecast shape:", df_forecast.shape)
df_forecast.head()

In [ ]:
def select_latest_forecast(target_time, horizon_hours, forecast_df):
    """Latest forecast with publishTime <= target_time - horizon_hours."""
    cutoff = target_time - pd.Timedelta(hours=horizon_hours)
    cand = forecast_df[(forecast_df["startTime"] == target_time) & (forecast_df["publishTime"] <= cutoff)]
    if cand.empty:
        return np.nan
    return cand.sort_values("publishTime", ascending=False).iloc[0]["generation"]

# Build merged dataset for horizon = 4h (as in the app)
HORIZON = 4
merged = []
for _, row in df_actual.iterrows():
    t = row["startTime"]
    pred = select_latest_forecast(t, HORIZON, df_forecast)
    merged.append({"startTime": t, "actual": row["generation"], "forecast": pred})
df_merged = pd.DataFrame(merged)
df_merged["error"] = df_merged["actual"] - df_merged["forecast"]
df_merged = df_merged.dropna(subset=["forecast"])
print("Merged (4h horizon) shape:", df_merged.shape)
df_merged.head(10)

## 2. Forecast error metrics

Mean, median, and p99 of the forecast error (actual - forecast).

In [ ]:
err = df_merged["error"]
print("Error statistics (4h horizon):")
print("  Mean error (bias):     ", err.mean().round(2), "MW")
print("  Median error:          ", err.median().round(2), "MW")
print("  P99 error (99th %ile): ", np.percentile(err, 99).round(2), "MW")
print("  MAE:                   ", np.abs(err).mean().round(2), "MW")
print("  RMSE:                  ", np.sqrt((err**2).mean()).round(2), "MW")

## 3. Error vs forecast horizon

Compute MAE for horizons 0, 4, 12, 24, 48 hours.

In [ ]:
horizons = [0, 4, 12, 24, 48]
mae_by_horizon = []
for h in horizons:
    merged_h = []
    for _, row in df_actual.iterrows():
        t = row["startTime"]
        pred = select_latest_forecast(t, h, df_forecast)
        if not np.isnan(pred):
            merged_h.append({"actual": row["generation"], "forecast": pred})
    if merged_h:
        df_h = pd.DataFrame(merged_h)
        mae = (df_h["actual"] - df_h["forecast"]).abs().mean()
        mae_by_horizon.append({"horizon_hours": h, "MAE": mae})
df_horizon = pd.DataFrame(mae_by_horizon)
df_horizon

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(df_horizon["horizon_hours"], df_horizon["MAE"], "o-", color="steelblue", linewidth=2)
plt.xlabel("Forecast horizon (hours)")
plt.ylabel("MAE (MW)")
plt.title("Forecast error (MAE) vs horizon")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Error by time of day

Average absolute error by hour of day (UTC).

In [ ]:
df_merged["hour"] = df_merged["startTime"].dt.hour
err_by_hour = df_merged.groupby("hour")["error"].agg(mean_error="mean", median_error="median", MAE=lambda x: np.abs(x).mean())
err_by_hour

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(err_by_hour.index, err_by_hour["MAE"], color="steelblue", alpha=0.8)
ax.set_xlabel("Hour of day (UTC)")
ax.set_ylabel("MAE (MW)")
ax.set_title("Forecast MAE by time of day")
ax.set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.show()

## 5. Wind power reliability – actual generation

How much wind generation can we **reliably** count on to meet demand?  
We look at the distribution of actual half-hourly wind generation (Jan 2024).

In [ ]:
gen = df_actual["generation"]
print("Actual wind generation (Jan 2024) – summary:")
print(gen.describe())
print("\nPercentiles (MW):")
for p in [5, 10, 25, 50, 75, 90, 95]:
    print(f"  P{p}: {np.percentile(gen, p):.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(gen, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Generation (MW)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of half-hourly wind generation")
axes[0].axvline(gen.quantile(0.1), color="red", linestyle="--", label="P10")
axes[0].axvline(gen.quantile(0.5), color="green", linestyle="--", label="P50")
axes[0].legend()
axes[1].plot(df_actual["startTime"], df_actual["generation"], alpha=0.7, color="steelblue")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Generation (MW)")
axes[1].set_title("Wind generation over January 2024")
plt.tight_layout()
plt.show()

## 6. Recommendation: reliable wind capacity

**Question:** How many MW of wind can we reliably expect to be available?

**Reasoning:**
- Wind is variable; we should not plan for the mean or peak.
- A conservative “reliable” level is one that is **exceeded most of the time** (e.g. 80–90%), so that we can count on at least this much for planning.
- **P10** (10th percentile): generation is above this value 90% of the time → a conservative “reliable” floor.
- **P25**: above this 75% of the time → a slightly less conservative option.

Using January 2024 actuals as evidence:

In [ ]:
p10 = np.percentile(gen, 10)
p25 = np.percentile(gen, 25)
p50 = np.percentile(gen, 50)
above_p10 = (gen >= p10).mean() * 100
above_p25 = (gen >= p25).mean() * 100
print(f"P10 = {p10:.0f} MW  →  generation ≥ this value {above_p10:.1f}% of the time")
print(f"P25 = {p25:.0f} MW  →  generation ≥ this value {above_p25:.1f}% of the time")
print(f"P50 (median) = {p50:.0f} MW")

**Recommendation:**  
For planning “reliable” wind availability to meet demand, we recommend using **approximately the P10 or P25 level** (e.g. **~1,500–2,500 MW** from this January sample, depending on required reliability).  
- Use **P10** if you need a very conservative floor (wind above this ~90% of the time).  
- Use **P25** if you accept a bit more risk (above this ~75% of the time).  

*Note: January is one month; a full-year analysis would give a more robust recommendation. Weather-dependent renewables always require backup or demand-side flexibility for the remaining risk.*